In [1]:
%%writefile preprocess_utils.py
import pandas as pd
import re
import numpy as np

# ==========================================
# 1. 斷詞 Worker (給程式一用)
# ==========================================
def worker_tokenization_task(df_chunk):
    """
    輸入: 包含 'title_clean' 的 DataFrame chunk
    輸出: 新增 'tokens_str' 欄位的 DataFrame chunk
    """
    # 延遲匯入 monpa，避免多核心卡死
    try:
        import monpa
    except ImportError:
        pass

    token_str_list = []
    
    # 防呆：確保轉為字串
    titles = df_chunk['title_clean'].astype(str)
    
    for text in titles:
        try:
            # 1. 斷詞
            words = monpa.cut(text)
            # 2. 過濾短詞與停用詞
            valid_words = [w for w in words if len(w) > 1 and w not in ['新聞', '報導', '表示', '曝光', '今日']]
            # 3. 接成空白分隔字串
            token_str_list.append(" ".join(valid_words))
        except:
            token_str_list.append("")
    
    df_chunk['tokens_str'] = token_str_list
    return df_chunk

# ==========================================
# 2. 數值特徵提取 (給程式二用，若需要現場算)
# ==========================================
def extract_viral_features(row):
    title = str(row['title_clean'])
    return pd.Series([
        len(title) if len(title) > 0 else 0, # title_len
        title.count('!') + title.count('！') + title.count('?') + title.count('？'), # punct_count
        1 if re.search(r'\d', title) else 0 # has_digit
    ], index=['title_len', 'punct_count', 'has_digit'])

Overwriting preprocess_utils.py


In [2]:
import pandas as pd
import numpy as np
import time
from multiprocessing import Pool, cpu_count

# 匯入工具
try:
    from preprocess_utils import worker_tokenization_task
except ImportError:
    print("❌ 錯誤：找不到 preprocess_utils.py")
    exit()

# 設定
INPUT_FILE = 'News_Data_cleaned.csv'   # 來源：先前清洗過的乾淨資料
OUTPUT_CSV = 'News_Data_tokenized.csv' # 輸出：準備給模型訓練用的資料
NUM_CORES = 12

if __name__ == '__main__':
    print(f"🚀 [Step 1] 啟動斷詞前處理 (Tokenization)...")
    start_time = time.time()
    
    # 1. 讀取資料
    try: 
        df = pd.read_csv(INPUT_FILE)
        print(f"   成功讀取 {INPUT_FILE}")
    except: 
        print(f"❌ 找不到 {INPUT_FILE}，請確認路徑！")
        exit()
        
    # 2. 平行斷詞
    print(f"   正在使用 {NUM_CORES} 核心進行 Monpa 斷詞...")
    df_split = np.array_split(df, NUM_CORES)
    
    with Pool(processes=NUM_CORES) as pool:
        results = pool.map(worker_tokenization_task, df_split)
    
    df_final = pd.concat(results, axis=0).reset_index(drop=True)
    
    # 3. 存檔
    print(f"💾 儲存含斷詞資料: {OUTPUT_CSV}")
    # 為了節省空間，只保留必要欄位
    keep_cols = ['label', 'title_clean', 'tokens_str', 'title_len', 'punct_count', 'has_digit']
    # 如果原本 CSV 沒有特徵欄位，程式二會自己算，但如果有就保留
    final_cols = [c for c in keep_cols if c in df_final.columns]
    
    df_final[final_cols].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    
    print(f"✅ 斷詞完成！耗時: {time.time() - start_time:.2f} 秒")
    print(f"   資料預覽:\n{df_final[['title_clean', 'tokens_str']].head(3)}")

🚀 [Step 1] 啟動斷詞前處理 (Tokenization)...
   成功讀取 News_Data_cleaned.csv
   正在使用 12 核心進行 Monpa 斷詞...


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.
+-------------------------------------------